In [ ]:
import collections
import re
import pandas as pd
from google.colab import files

# ==========================================
# 1. DATA ACQUISITION & PREPROCESSING
# ==========================================

# Convert Google Drive sharing link to a direct download URL
url = "https://drive.google.com/file/d/1sl621sxq4AalmviriC15STZxrPAZdGU3/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]
product_category_df = pd.read_csv(path)

# Sanitize the 'type' column to avoid trailing spaces/formatting issues
product_category_df["type"] = product_category_df["type"].astype(str).str.strip()

# Combine name and description into a unified text column optimized for case-insensitive keyword filtering
product_category_df["text_corpus"] = (
    product_category_df["name"].fillna("") + " " + product_category_df["desc"].fillna("")
).str.lower()


# ==========================================
# 2. STRATEGY CONFIGURATION (RULES & SPECS)
# ==========================================

# Identify the top 30 most common product types to prioritize matching
top_30_types = product_category_df["type"].value_counts().head(30).index.tolist()

# Direct lookup for main device hardware types
HARDWARE_MAP = {
    "1714": "iPad Hardware", "106431714": "iPad Hardware", "51861714": "iPad Hardware",
    "24861714": "iPad Hardware", "12141714": "iPad Hardware", "13621714": "iPad Hardware",
    "11821715": "iPod Hardware",
    "1716": "iPhone Hardware", "24821716": "iPhone Hardware", "85651716": "iPhone Hardware",
    "51601716": "iPhone Hardware", "85641716": "iPhone Hardware", "113291716": "iPhone Hardware",
    "24811716": "iPhone Hardware", "21561716": "iPhone Hardware", "113281716": "iPhone Hardware",
    "24885185": "Apple Watch Hardware", "24895185": "Apple Watch Hardware",
    "5,74E+15": "Mac Hardware", "21632158": "Mac Hardware", "5,43E+15": "Mac Hardware",
    "118692158": "Mac Hardware", "51882158": "Mac Hardware",
    "1,02E+12": "Macbook Hardware", "5,39E+11": "Macbook Hardware", "2158": "Macbook Hardware",
    "2,17E+11": "Macbook Hardware",
    "1282": "Macbook Hardware"
}

# Words to ignore during fallback keyword extraction
stop_words = {
    "and", "the", "for", "with", "apple", "macbook", "iphone", "ipad", "pro",
    "inch", "case", "black", "white", "to", "in", "of", "compatible", "support",
    "your", "new", "design", "from", "this", "that", "its", "plus", "inches", "retina",
    "core", "premium", "microfiber", "gb", "tb", "mhz", "ddr", "usb", "mac", "mini", "air"
}

# Regex mapping logic matching common product markers to human-readable categories
category_rules = [
    (r"nas|synology|qnap|server", "Network Storage (NAS)"),
    (r"ram|memory|dimm|so-dimm", "Computer Memory"),
    (r"ssd|solid state", "Internal SSD Storage"),
    (r"drive|hard|external|hdd", "External Storage Drives"),
    (r"case|cover|sleeve|leather|silicone|shell|backpack|bag", "Protective Cases & Bags"),
    (r"cable|adapter|hub|dock|thunderbolt|charger|power|charging", "Cables & Connectivity"),
    (r"repair|screen repair|battery repair|labor", "Technical Repair Services"),
    (r"headset|headphone|speaker|audio|bluetooth", "Audio Equipment"),
    (r"keyboard|mouse|pointer|stylus|pen", "Input Peripherals"),
    (r"monitor|display|screen", "Monitors & Displays"),
    (r"strap|band|loop|watch band|leather loop", "Watch Accessories"),
    (r"link|switch|router|access point|mesh", "Networking Equipment")
]

# CATEGORY-SPECIFIC ECONOMIC THRESHOLDS
# Format -> "Category Name": (Budget_Upper_Limit, MidRange_Upper_Limit)
PRICE_THRESHOLDS = {
    "Network Storage (NAS)": (150, 400),
    "Computer Memory": (30, 80),
    "Internal SSD Storage": (60, 150),
    "External Storage Drives": (50, 130),
    "Protective Cases & Bags": (25, 65),
    "Cables & Connectivity": (20, 50),
    "Technical Repair Services": (60, 130),
    "Audio Equipment": (40, 150),
    "Input Peripherals": (35, 90),
    "Monitors & Displays": (200, 500),
    "Mac Hardware": (500, 1500),
    "Macbook Hardware": (600, 1500),
    "iPhone Hardware": (250, 650),
    "iPad Hardware": (200, 500),
    "Apple Watch Hardware": (150, 350),
    "iPod Hardware": (40, 120),
    "Watch Accessories": (20, 60),          # Budget < €20, Mid < €60, Premium >= €60
    "Networking Equipment": (80, 250),       # Budget < €80, Mid < €250, Premium >= €250
    "Default": (40, 150)
}

# Runtime tracking storage for group mapping
category_mapping = {}

print("--- Step 1: Mapping Categories to Product Types ---")

# ==========================================
# 3. RULE MATCHING & ALGORITHMIC FALLBACK
# ==========================================

for prod_type in top_30_types:
    clean_key = str(prod_type).replace(".0", "")
    type_df = product_category_df[product_category_df["type"] == prod_type]

    # PRIORITY 1: Direct ID Map Override
    if clean_key in HARDWARE_MAP:
        winner_category = HARDWARE_MAP[clean_key]

    # PRIORITY 2: Text Majority Vote Fallback
    else:
        counts = {name: 0 for _, name in category_rules}
        counts["General Accessories"] = 0

        for _, row in type_df.iterrows():
            matched = False
            for pattern, name in category_rules:
                if re.search(pattern, row["text_corpus"]):
                    counts[name] += 1
                    matched = True
                    break
            if not matched:
                counts["General Accessories"] += 1

        winner_category = max(counts, key=counts.get)

        if winner_category == "General Accessories":
            all_words = re.findall(r"\b[a-z]{3,}\b", " ".join(type_df["text_corpus"]))
            filtered_words = [w for w in all_words if w not in stop_words]

            top_keyword = collections.Counter(filtered_words).most_common(1)
            if top_keyword:
                winner_category = f"General {top_keyword[0][0].title()} Accessories"

    # Store group classification decision
    category_mapping[clean_key] = winner_category
    print(f"Type ID: {clean_key:12} -> Assigned Category: {winner_category}")


# ==========================================
# 4. ROW-BY-ROW INDIVIDUAL PRODUCT PRICING
# ==========================================

print("\n--- Step 2: Applying Price Tiers Row-by-Row per Product ---")

# Apply the master category map back to every product row first
# Clean up temporary string decimal representations to guarantee map matching consistency
product_category_df["clean_type_key"] = product_category_df["type"].astype(str).str.replace(".0", "", regex=False)
product_category_df["category"] = product_category_df["clean_type_key"].map(category_mapping).fillna("Other")

# Helper function to evaluate each product individually based on its own price tag
def evaluate_individual_tier(row):
    assigned_cat = row["category"]
    individual_price = row["price"]

    # Extract unique financial boundaries for this specific row's category
    budget_limit, mid_limit = PRICE_THRESHOLDS.get(assigned_cat, PRICE_THRESHOLDS["Default"])

    if individual_price < budget_limit:
        return "Budget"
    elif individual_price < mid_limit:
        return "Mid-Range"
    else:
        return "Premium"

# Apply individual pricing logic to every row independently
product_category_df["subcategory"] = product_category_df.apply(evaluate_individual_tier, axis=1)


# ==========================================
# 5. WORKSPACE CLEANUP & EXPORT
# ==========================================

# Drop processing helper columns
product_category_df = product_category_df.drop(columns=["text_corpus", "clean_type_key"])

# Save and trigger download
product_category_df.to_csv("products_categorized.csv", index=False)
files.download("products_categorized.csv")

print("\nSuccess! Individual classifications complete. Downloaded as 'products_categorized.csv'")

--- Step 1: Mapping Categories to Product Types ---
Type ID: 11865403     -> Assigned Category: Protective Cases & Bags
Type ID: 12175397     -> Assigned Category: Network Storage (NAS)
Type ID: 1298         -> Assigned Category: Input Peripherals
Type ID: 11935397     -> Assigned Category: External Storage Drives
Type ID: 11905404     -> Assigned Category: General Ipod Accessories
Type ID: 1282         -> Assigned Category: Macbook Hardware
Type ID: 12635403     -> Assigned Category: Protective Cases & Bags
Type ID: 13835403     -> Assigned Category: Protective Cases & Bags
Type ID: 5,74E+15     -> Assigned Category: Mac Hardware
Type ID: 1364         -> Assigned Category: Computer Memory
Type ID: 12585395     -> Assigned Category: Cables & Connectivity
Type ID: 1296         -> Assigned Category: Monitors & Displays
Type ID: 1325         -> Assigned Category: Cables & Connectivity
Type ID: 5384         -> Assigned Category: Audio Equipment
Type ID: 1433         -> Assigned Category: I

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Success! Individual classifications complete. Downloaded as 'products_categorized.csv'
